# Practice 2b: Group-level statistics on the Weimar Jazz Database

---

Welcome back. In **Practice 2a** we:
- loaded the feature table and the metadata,
- described the corpus,
- looked at features over time (coloured by performer),
- built a correlation heatmap,
- ran a **two-sample t-test** comparing Parker and Davis on `event_density`,
- and cross-checked the features against the original MIDI files with music21.

In this notebook we move from **pairwise** comparisons to **group-level** statistics. Each part follows the same rhythm: a **research question**, the right **test**, **assumption checks**, the test statistic, the **p-value** and an **effect size**, and finally **interpretation**.

**What you will do in this notebook**

1. Compare **all jazz styles** with a **one-way ANOVA** on `avgtempo`.
2. Ask whether jazz solos have become **more complex over time** using **correlation**.
3. Test whether **tonal approach** (blues / functional / modal) is **associated with style** using a **chi-squared test**.
4. A short recap: which test when?


## Part 1: Setup (Colab)

Run the cell below once at the start of the session. It installs the Python packages we need.


In [ ]:
# Install packages (Colab / first run). Safe to run again.
%pip install pandas numpy matplotlib seaborn scipy statsmodels --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

RNG_SEED = 42
np.random.seed(RNG_SEED)

BG = "#F7F7F7"
STYLE_PALETTE = {
    "TRADITIONAL": "#8B7355",
    "SWING":       "#FFC000",
    "BEBOP":       "#C00000",
    "COOL":        "#AFCDCA",
    "HARDBOP":     "#7F3FBF",
    "POSTBOP":     "#3A3A3A",
    "FREE":        "#E07A5F",
}
plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "axes.edgecolor": "#888888",
    "axes.labelcolor": "#222222",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "grid.color": "#CCCCCC",
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
})

print("Libraries loaded.")

In [ ]:
# Reload the data (in case you are running this notebook independently of 2a)
import urllib.request

url = "https://raw.githubusercontent.com/aljanaki/Digital_musicology/50f011b927b159c756979d5126d04c30a0ff43fa/Practice%20sessions/Practice%202/Weimars_solos_with_features_and_metadata.csv"
urllib.request.urlretrieve(url, "data.csv")

df = pd.read_csv("data.csv")
print("Table:", df.shape)

---
## Part 2: One-way ANOVA — tempo across jazz styles

### Research question

> **Does the average performance tempo differ across the six main jazz styles?**

A **t-test** compares **two** groups. When we have **more than two**, we use a **one-way ANOVA** (Analysis of Variance). ANOVA asks a single pooled question:

- **H₀:** All style means are equal.
- **H₁:** At least one style mean is different.

If ANOVA is significant, we follow up with **Tukey's HSD** (Honestly Significant Difference) to see **which pairs** of styles actually differ. This is essential: "some styles differ" is not a satisfying finding — we want to know *which ones*.

**Effect size** for ANOVA: **eta-squared** (η²) = fraction of total variance explained by the grouping.


In [ ]:
# Build one list of values per style (filter out rare FREE and any NaNs)
style_order = ["TRADITIONAL", "SWING", "BEBOP", "COOL", "HARDBOP", "POSTBOP"]
groups = [df.loc[df["style"] == s, "avgtempo"].dropna().values for s in style_order]

for s, g in zip(style_order, groups):
    print(f"{s:12s} n = {len(g):3d}   mean tempo = {g.mean():6.1f}   std = {g.std(ddof=1):5.1f}")

In [ ]:
# Quick assumption check: Levene (equal variances across groups)
w_lev, p_lev = stats.levene(*groups)
print(f"Levene (homogeneity of variance): W = {w_lev:.3f}, p = {p_lev:.3g}")

# The ANOVA itself
F_stat, p_anova = stats.f_oneway(*groups)
print(f"ANOVA: F = {F_stat:.3f}, p = {p_anova:.3g}")

# Eta-squared
all_vals = np.concatenate(groups)
grand_mean = all_vals.mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
ss_total = ((all_vals - grand_mean)**2).sum()
eta2 = ss_between / ss_total
print(f"Eta-squared = {eta2:.3f}   "
      f"({'small' if eta2 < 0.06 else 'medium' if eta2 < 0.14 else 'large'} effect)")

In [ ]:
# Tukey HSD — which pairs of styles differ?
tmp = df[df["style"].isin(style_order)][["style", "avgtempo"]].dropna()
res = pairwise_tukeyhsd(endog=tmp["avgtempo"], groups=tmp["style"], alpha=0.05)
print(res)

fig = res.plot_simultaneous()
plt.title("Tukey HSD — mean tempo by style")
plt.tight_layout()
plt.show()

### How to read Tukey HSD

Each row of the table is a pair of styles. `meandiff` is the difference of the two group means; `p-adj` is the p-value **adjusted for multiple comparisons**; `reject` is `True` when we can say the two styles really differ.

The **plot** shows each style's 95% confidence interval for the mean. **If two intervals do not overlap**, those styles differ significantly — a quick visual summary of the whole table.

Is **Bebop** faster than **Traditional** and **Swing**? What about Bebop and later styles (Cool, Hardbop, Postbop)?


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 1 — Another ANOVA</b><br><br>
Run a one-way ANOVA for <code>int_entropy</code> (interval entropy — how varied the interval choices are) across the six main styles. Is there a significant difference? Which pairs differ according to Tukey?
</div>

---
## Part 3: Correlation — has jazz become more complex over time?

### Research question

> **Is there a relationship between the year a solo was recorded and its pitch complexity?**

This is one of the classic **historical** questions in computational jazz studies. Previous analyses of the WJazzD have reported that several complexity features tend to increase with recording year — jazz soloists in the postbop era use a wider, more varied pitch vocabulary than in the swing era.

We will test this with **correlation** between `recordingyear` and `pitch_entropy`.

- **Pearson r** measures **linear** relationship.
- **Spearman ρ** measures **monotonic** relationship based on ranks (more robust to outliers).

When they agree you can be more confident the relationship is real. When they differ noticeably, the relationship is likely **nonlinear** or driven by outliers.

**Remember:** Correlation is *not* causation. A correlation with time can reflect many underlying mechanisms — changes in instruments, recording technology, audience expectations, style conventions — not a unified "jazz got more complex" story.


In [ ]:
# Keep only rows where both variables are present
valid = df[["recordingyear", "pitch_entropy"]].dropna()
print(f"Usable solos: {len(valid)}")

r_p, p_p = stats.pearsonr(valid["recordingyear"],  valid["pitch_entropy"])
r_s, p_s = stats.spearmanr(valid["recordingyear"], valid["pitch_entropy"])
print(f"Pearson:  r = {r_p:+.3f}   p = {p_p:.3g}")
print(f"Spearman: ρ = {r_s:+.3f}   p = {p_s:.3g}")

In [ ]:
# Scatter plot with a linear fit
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(valid["recordingyear"], valid["pitch_entropy"],
           alpha=0.4, s=20, color="#2A2A64")
m, b = np.polyfit(valid["recordingyear"], valid["pitch_entropy"], 1)
xs = np.linspace(valid["recordingyear"].min(), valid["recordingyear"].max(), 100)
ax.plot(xs, m*xs + b, color="#DE5B59", linewidth=2,
        label=f"Linear fit (Pearson r = {r_p:+.2f})")
ax.set_xlabel("Recording year")
ax.set_ylabel("Pitch entropy")
ax.set_title("Pitch entropy over the history of jazz (1925–2009)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Interpretation

In this corpus the correlation is typically **weakly positive** — jazz *has* gotten somewhat more pitch-varied over time, but the cloud is wide. The `r` value tells you the strength:

| `r` | Rough interpretation |
|-------|----------------------|
| 0.0 – 0.1 | Essentially no relationship |
| 0.1 – 0.3 | Weak |
| 0.3 – 0.5 | Moderate |
| 0.5 – 0.7 | Strong |
| 0.7 – 1.0 | Very strong |

The **p-value** tells you whether the observed correlation is unlikely under the null hypothesis of no true linear/monotonic relationship. With ~450 data points even weak correlations will be highly significant — so pay attention to the **size** of `r`, not just the p-value.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 2 — A different feature over time</b><br><br>
Repeat this correlation analysis for <code>int_entropy</code> (how varied the interval choices are). Is the trend stronger or weaker than for pitch entropy? Also check <code>ratio_chromatic_sequences</code> — did jazz become more chromatic over time?
</div>

---
## Part 4: Chi-squared — style × tonality type

### Research question

> **Is the distribution of tonality types (functional, blues, modal, free) associated with jazz style?**

Both variables are **categorical**:

- `style`: historical period of the solo (TRADITIONAL, SWING, BEBOP, COOL, HARDBOP, POSTBOP).
- `tonality_type`: the general harmonic approach — `FUNCTIONAL` (standard tonal harmony with ii–V–I progressions), `BLUES` (blues form), `MODAL` (static-scale / modal harmony, think *Kind of Blue* or *A Love Supreme*), `FREE` (free jazz).

Musicologically we **expect** modal solos to cluster in **Postbop** (Davis's *Kind of Blue* from 1959 is sometimes taken as the birth of modal jazz), and blues to be roughly evenly spread across styles.

- **H₀:** Style and tonality type are **independent**.
- **H₁:** They are **associated**.

The test: **Pearson's chi-squared**. Effect size: **Cramér's V**.

**Watch out:** chi-squared becomes unreliable if too many **expected** cell counts are below 5. We will check that.

In [ ]:
# Build the contingency table
ct = pd.crosstab(df["style"], df["tonality_type"])
# Reorder rows chronologically
ct = ct.reindex(["TRADITIONAL", "SWING", "BEBOP", "COOL", "HARDBOP", "POSTBOP"])
print("Observed counts:")
print(ct)

In [ ]:
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
print(f"Chi-squared statistic: {chi2:.3f}")
print(f"Degrees of freedom:    {dof}")
print(f"p-value:               {p_chi:.3g}")
print()
print("Expected counts under H0 (independence):")
print(np.round(expected, 1))
print()
print(f"All expected ≥ 5? {bool(np.all(expected >= 5))}")

# Cramér's V — effect size
n = ct.values.sum()
r, c = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r, c) - 1)))
print(f"Cramér's V = {cramers_v:.3f}")
print("(Rule of thumb: 0.1 small, 0.3 medium, 0.5 large)")

In [ ]:
# Heatmap of the observed counts — easier to read than a raw table
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(ct, annot=True, fmt="d", cmap="YlOrBr", cbar=True, ax=ax)
ax.set_title("Style × Tonality type (observed counts)")
ax.set_xlabel("Tonality type")
ax.set_ylabel("Style")
plt.tight_layout()
plt.show()

### Interpretation

If `p_chi` is very small and Cramér's V is borderline medium (~0.3), we have some evidence that **style and tonality are associated** — certain styles concentrate certain tonal approaches.

What can you say about harmony use and style based on what you found?

How is Postbop different from other styles?
What can we say about blues tonality use over time?

**Important caveat:** chi-squared tells you *that* there is an association, not *which cells drive it*. To inspect that, look at **residuals** (observed – expected) per cell — positive residuals show where a style has *more* of a tonality than expected under independence.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 3 — Style × rhythm feel</b><br><br>
Build <code>pd.crosstab(df['style'], df['rhythmfeel'])</code> and run a chi-squared test. Rhythm feel can be SWING, LATIN, EVEN, etc. Are some rhythmic feels associated with specific historical styles?<br><em>Tip:</em> First look at <code>df['rhythmfeel'].value_counts()</code> — if a category has very few solos, the expected-count rule may be violated.
</div>

---
## Part 5: Summary — which test when?

| Question type | Variables | Test used |
|---------------|-----------|-----------|
| Compare **two** group means | event_density × performer (Parker vs. Davis) | **t-test** (+ Cohen's d) — Practice 2a |
| Compare **3+** group means | avgtempo × style | **One-way ANOVA** + Tukey |
| Linear/monotonic relationship of two numeric variables | pitch_entropy × recordingyear | **Pearson / Spearman** |
| Association of two categorical variables | style × tonality_type | **Chi-squared** + Cramér's V |

**Practical notes to take home**

- With 456 solos, **tiny differences produce tiny p-values** — always look at **effect size** (Cohen's d, η², Cramér's V, r).
- Always **check assumptions** before trusting a test: Shapiro–Wilk for normality, Levene for equal variances.
- **Correlation is not causation.** A historical trend could be driven by many confounded variables (instruments, recording quality, style conventions all changed together).
- **Musicological plausibility matters.** A highly significant result that contradicts everything we know about the music is more likely to reflect a data or analysis issue than a real discovery.

---

### Homework

The homework (Homework 2) will ask you to run a parallel set of analyses on the same dataset — different performers, different features, different questions — and to write a short interpretation of each result. Good luck!


---
## Further resources

Short, accessible resources for each statistical topic in this notebook. Mix of video, interactive tools, and text — pick whatever format clicks for you.

| Topic | Resource | Format | Link |
|-------|----------|--------|------|
| **All topics** | Seeing Theory (Brown University) — interactive probability & statistics | Interactive | [seeing-theory.brown.edu](https://seeing-theory.brown.edu/) |
| **p-values** | StatQuest — *p-values: What they are and how to interpret them* | Video (11 min) | [youtube.com](https://www.youtube.com/watch?v=vemZtEM63GY) |
| **Hypothesis testing** | StatQuest — *Hypothesis Testing and The Null Hypothesis* | Video (15 min) | [youtube.com](https://www.youtube.com/watch?v=0oc49DyA3hU) |
| **ANOVA** | Khan Academy — *Analysis of Variance (ANOVA)* | Video + exercises | [khanacademy.org](https://www.khanacademy.org/math/statistics-probability/analysis-of-variance-anova-library) |
| **ANOVA** | Crash Course Statistics #33 — *ANOVA* | Video (13 min) | [youtube.com](https://www.youtube.com/watch?v=oOuu8IBd-yo) |
| **Correlation** | StatQuest — *Pearson's Correlation, Clearly Explained* | Video (19 min) | [youtube.com](https://www.youtube.com/watch?v=xZ_z8KWkhXE) |
| **Chi-squared** | Khan Academy — *Chi-square tests for categorical data* | Video + exercises | [khanacademy.org](https://www.khanacademy.org/math/statistics-probability/inference-categorical-data-chi-square-tests) |
| **Chi-squared** | Math is Fun — *Chi-Square Test* | Text (step-by-step) | [mathsisfun.com](https://www.mathsisfun.com/data/chi-square-test.html) |
| **Effect size** | StatQuest — *Clearly Explained* playlist (Cohen's d, R²) | Video series | [statquest.org](https://statquest.org/video-index/) |